In [ ]:
# run_gpr_analysis_v7_no_dotenv.py

import os
from typing import List, Dict, Optional

# NOTE: The 'dotenv' dependency has been removed.
# from dotenv import load_dotenv 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

# --- Step 1: Define the "Three-Part Variable" Custom Risk Events ---
CUSTOM_RISK_EVENTS = [
    {
        "event_name": "China_WTO_Accession_Impact",
        "event_timeframe": "Late 2001 and subsequent years",
        "event_description": (
            "Analyze the text for discussions related to the consequences of China's accession to the WTO. "
            "Look for concepts like shifts in global manufacturing, supply chain relocations, increased competition in specific markets, "
            "and changes in international trade dynamics, even if 'WTO' is not explicitly mentioned."
        )
    },
    {
        "event_name": "US_China_Chip_Export_Controls",
        "event_timeframe": "October 2022 and onwards",
        "event_description": (
            "Analyze the text for impacts stemming from heightened US-China tech tensions, specifically regarding semiconductors. "
            "Look for discussions of supply chain disruptions for advanced computing components, risks in electronics manufacturing, "
            "the need to redesign products, or the impact of export controls on technology access."
        )
    }
]

# --- Step 2: Define the Pydantic model to include the detailed scoring ---
class RiskAnalysis(BaseModel):
    """A structured analysis for a single event, including the detailed scoring breakdown."""
    event_name: str = Field(description="The unique name of the event that was analyzed.")
    event_timeframe: str = Field(description="The timeframe of the event, provided as input.")
    baseline_score: int = Field(description="The initial baseline score selected (0, 1, 21, 51, or 81).")
    adjustment_points: int = Field(description="Points added or subtracted from the baseline.")
    final_score: int = Field(description="The final calculated score (baseline + adjustment).", ge=0, le=100)
    reasoning: str = Field(description="A detailed causal argument, which MUST explain the choice of baseline and the reason for the adjustment points.")
    extracted_sentences: List[str] = Field(default=[], description="A list of original sentences from the text that are most relevant to this event's impact.")

# --- Step 3: Define the powerful "Baseline + Adjust" Prompt ---
prompt_template = """
You are a hyper-precise risk analyst. Your task is to analyze a document based on a SINGLE specific event description provided below. You must follow all rules meticulously.

**EVENT TO ANALYZE:**
* **Event Name:** {event_name}
* **Event Timeframe:** {event_timeframe}
* **Analysis Focus:** {event_description}

---
**GOLDEN RULE #1: THE ZERO SCORE**
If, after scanning the text, the concepts related to the `Analysis Focus` are COMPLETELY ABSENT, the score MUST be exactly 0. In this case, `baseline_score`, `adjustment_points`, and `final_score` must all be 0. Do NOT assign a score of 1. THIS IS YOUR MOST IMPORTANT RULE.

---
**SCORING METHODOLOGY: "Baseline and Adjust"**
For any risk that IS present, you must follow this two-step calculation:

1.  **Select a Baseline:** Choose a `baseline_score` from the list below. This is the STARTING POINT.
    * **Baseline 1:** For vague, indirect, or minor conceptual links to the event.
    * **Baseline 21:** For direct discussion of the event's concepts, but with low, hypothetical, or non-material impact.
    * **Baseline 51:** For significant discussion where the event's impact is described as direct and potentially material.
    * **Baseline 81:** For discussion where the event's impact is presented as a central, severe, and strategic factor for the company.

2.  **Calculate Adjustment:** Based on textual evidence, determine the `adjustment_points`.
    * **Add points** for aggravating factors (e.g., specific negative examples are given, repeated mentions, quantified financial impact).
    * **Subtract points** for mitigating factors (e.g., strong countermeasures described, impact is presented as well-managed).
    * The final score MUST stay within a logical range for the chosen baseline.

3.  **Calculate Final Score:** `final_score` = `baseline_score` + `adjustment_points`.

---
**FEW-SHOT EXAMPLES (How your brain should work):**

* **Example for a non-zero, adjusted score:**
    * *Input Event:* US_China_Chip_Export_Controls
    * *Thought Process:* "The text discusses supply chain risks for 'advanced microprocessors' (Baseline 51). It also mentions a $50M increase in R&D to redesign products, a clear aggravating factor. I will add 18 points."
    * *JSON Output Snippet for this risk:*
        ```json
        {{
            "event_name": "US_China_Chip_Export_Controls",
            "event_timeframe": "October 2022 and onwards",
            "baseline_score": 51,
            "adjustment_points": 18,
            "final_score": 69,
            "reasoning": "Baseline is 51, as the text directly discusses supply chain risks for key components. Adjustment of +18 is due to the specific, quantified $50M R&D investment mentioned as a direct consequence, indicating high materiality.",
            "extracted_sentences": ["Our supply chain for advanced microprocessors faces significant uncertainty...", "We have allocated an additional $50 million to R&D for product redesigns..."]
        }}
        ```

* **Example for a zero score:**
    * *Input Event:* China_WTO_Accession_Impact
    * *Thought Process:* "I have scanned the document for concepts like 'global manufacturing shifts', 'increased competition from Asia', or 'WTO'. There are no such discussions. The risk is absent."
    * *JSON Output Snippet for this risk:*
        ```json
        {{
            "event_name": "China_WTO_Accession_Impact",
            "event_timeframe": "Late 2001 and subsequent years",
            "baseline_score": 0,
            "adjustment_points": 0,
            "final_score": 0,
            "reasoning": "The document contains no discussion of concepts related to China's WTO accession or its long-term impact on trade and competition. Per the Golden Rule, the score is 0.",
            "extracted_sentences": []
        }}
        ```
---

**YOUR TASK:**
Now, apply this exact methodology to the `DOCUMENT TEXT` below for the specified event. Produce a single, valid JSON object according to the required format.

**DOCUMENT TEXT (Item 1a):**
```text
{document_text}
```

**REQUIRED JSON OUTPUT FORMAT:**
{format_instructions}
"""

def run_full_analysis():
    """
    Main execution function to run the final, complete analysis pipeline.
    """
    # --- Check for API Key directly from environment variables ---
    if not os.getenv("OPENAI_API_KEY"):
        print("Error: OPENAI_API_KEY is not set. Please set it as an environment variable before running the script.")
        return

    # --- Initialize Model and Parser ---
    llm = ChatOpenAI(model="gpt-4o", temperature=0.0, model_kwargs={"response_format": {"type": "json_object"}})
    parser = JsonOutputParser(pydantic_object=RiskAnalysis)
    
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["document_text", "event_name", "event_timeframe", "event_description"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    chain = prompt | llm | parser

    # --- Load Document ---
    try:
        with open("item1a_text.txt", "r", encoding="utf-8") as f:
            item1a_content = f.read()
    except FileNotFoundError:
        print("Error: `item1a_text.txt` not found. Please create this file in the same directory as the script.")
        return

    # --- Loop through each event and run analysis ---
    all_results = []
    print(f"--- Starting Analysis for {len(CUSTOM_RISK_EVENTS)} Custom Events... ---")
    for i, event in enumerate(CUSTOM_RISK_EVENTS, 1):
        print(f"\n({i}/{len(CUSTOM_RISK_EVENTS)}) Analyzing Event: {event['event_name']}...")
        try:
            result = chain.invoke({
                "document_text": item1a_content,
                "event_name": event["event_name"],
                "event_timeframe": event["event_timeframe"],
                "event_description": event["event_description"]
            })
            all_results.append(result)
            print(f"  > Analysis complete. Final Score: {result.get('final_score', 'N/A')}")
        except Exception as e:
            print(f"  > An error occurred during analysis for this event: {e}")
    
    # --- Print the final summary report ---
    print_summary_report(all_results)


def print_summary_report(results: List[Dict]):
    """
    Helper function to print the summary report for all analyzed events.
    """
    if not results:
        print("\nNo analysis results to display.")
        return

    print("\n\n======================================================")
    print("        CUSTOM RISK EVENT ANALYSIS - SUMMARY REPORT      ")
    print("======================================================")
    
    sorted_results = sorted(results, key=lambda x: x.get('final_score', 0), reverse=True)

    for result in sorted_results:
        print(f"\n--- Event: {result.get('event_name', 'N/A')} ({result.get('event_timeframe', 'N/A')}) ---")
        print(f"  Final Score (0-100): {result.get('final_score', 'N/A')}")
        print(f"  Calculation: {result.get('baseline_score', 'N/A')} (Baseline) + {result.get('adjustment_points', 'N/A')} (Adjustment)")
        print(f"  Reasoning: {result.get('reasoning', 'N/A')}")
        
        sentences = result.get('extracted_sentences', [])
        if sentences:
            print("  Supporting Evidence (Original Sentences):")
            for i, sentence in enumerate(sentences, 1):
                print(f'    {i}. "{sentence}"')
        else:
            print("  Supporting Evidence: None extracted.")
            
    print("\n======================================================")


if __name__ == "__main__":
    run_full_analysis()


--- Running Advanced GPR Risk Analysis with GPT-4o... This may take a moment. ---
--- Analysis Complete. Formatting Report... ---

      GEOPOLITICAL RISK (GPR) REPORT (V2)      

--- Risk Category: COVID_19_Pandemic ---
  Final Score (0-100): 61
  Calculation: 51 (Baseline) + 10 (Adjustment)
  Causal Reasoning: Baseline set to 51 due to direct mention of COVID-19 affecting advertising revenues and financial results. Adjustment of +10 points is added because the text discusses the pandemic's impact on revenue growth and market volatility, indicating a significant material impact.

--- Risk Category: Russia_Ukraine_Conflict ---
  Final Score (0-100): 0
  Calculation: 0 (Baseline) + 0 (Adjustment)
  Causal Reasoning: The document contains no direct or indirect mentions of the Russia-Ukraine conflict. Per the Golden Rule, the score is 0.

--- Risk Category: September_11_Attacks ---
  Final Score (0-100): 0
  Calculation: 0 (Baseline) + 0 (Adjustment)
  Causal Reasoning: The document conta

In [6]:
RISK_CATEGORIES: Dict[str, str] = {
    "Russia_Ukraine_Conflict": "Mentions of the Russia-Ukraine conflict, including its timeline, specific events, and its impact on industries such as supply chains, energy, or finance.",
    "COVID_19_Pandemic": "Mentions of the COVID-19 pandemic, its general timeline (e.g., outbreaks, lockdowns), and its effects on industries like healthcare, travel, labor, or manufacturing.",
    "September_11_Attacks": "Mentions of the September 11th, 2001 attacks, their specific timing, and any described long-term or comparative impacts on industries, particularly aviation, security, and insurance. This can act as a control case for historical risk discussion.",
}

# --- Step 3: Define the Desired, Structured Output Format (Pydantic Models) ---
class RiskAnalysis(BaseModel):
    """Analysis for a single risk category, with explicit calculation fields."""
    category: str = Field(description="The GPR risk category being analyzed.")
    baseline_score: int = Field(description="The initial baseline score selected (0, 1, 21, 51, or 81).")
    adjustment_points: int = Field(description="Points added or subtracted from the baseline. Can be positive or negative.")
    final_score: int = Field(description="The final calculated score (baseline + adjustment).", ge=0, le=100)
    reasoning: str = Field(description="The detailed causal argument, which MUST explain the choice of baseline and the reason for the adjustment points.")

class HighestRiskEvidence(BaseModel):
    """Evidence for the highest-scored risk category."""
    category: str = Field(description="The name of the category with the highest risk score.")
    relevant_sentences: List[str] = Field(description="A list of the most relevant original sentences from the source text that support the high score.")

class GPRFinalReport(BaseModel):
    """The final, complete GPR risk report with explicit calculations."""
    risk_analysis: List[RiskAnalysis] = Field(description="A list of analyses for all 8 GPR categories.")
    highest_risk_evidence: HighestRiskEvidence = Field(description="Evidence for the single highest-scored risk.")

# --- Step 4: Initialize the Model and Output Parser ---

# llm = ChatOpenAI(model="gpt-4o", temperature=0.0, model_kwargs={"response_format": {"type": "json_object"}})
parser = JsonOutputParser(pydantic_object=GPRFinalReport)
with open('test.txt', 'w') as f:
    f.write(parser.get_format_instructions())
# parser.get_format_instructions()